### **API Calls and Response structured in Class and sub-class**

**_Volume path for Binaries_**: /Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi

Run cell 4 & cell 5 to initialize first

In [0]:
from __future__ import annotations
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col as _col


In [0]:
# # Query to scope Audit reports for the 16.5K 
# query = """
# with full_list as (
#   select distinct cms1.Document_Id, ud1.BU from 
#   dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
#   left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
#   on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
#   left join
#   dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
#   on upper(trim(ud1.user_ID)) = upper(trim(
#     case 
#       when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
#       else cms1.created
#     end
#   ))

#   Union all
#   select distinct cms1.Document_Id, ud1.BU from 
#   dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
#   left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
#   on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
#   left join
#   dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
#   on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
# ),
# scanned_record as (
#   select *
# from (
#   select *,
#     row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
#   from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
# )
# where rn = 1
# ),
# inscope_list as(
#   select distinct Document_Id from full_list
#   where 
#   Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
#   and 
#   Document_Id not in (select distinct Document_Id from scanned_record where document_name is not null)
# ), 
# Audit_data as (
#   select
#     UA1.*,
#     case
#       when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
#       else 'Viewer_events'
#     end as usage_category,
#     UP1.BU
#   from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
#   left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
#   on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
# ),
# Audit_profile as (
#     SELECT
#     WEBI_Doc_ID,
#     COUNT(DISTINCT UserName) AS user_count,
#     COUNT(DISTINCT BU) AS bu_count
#     FROM Audit_data
#     GROUP BY WEBI_Doc_ID
# )
# select distinct Document_Id as BO_REPORTS  from inscope_list as l1 left join Audit_profile as a1
# on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID)
# where a1.WEBI_Doc_ID is not null
# and a1.user_count = 1
# limit 150
# """


# Find more schedule active reports
query="""
  select distinct Document_Id as BO_REPORTS  from  dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as wb2
  where wb2.Active_Schedule = true 
  and wb2.document_id not in (select distinct document_id from (
    select *,
      row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
  )
  where rn = 1
  )
"""
df = spark.sql(query)
webi_ids = [row.BO_REPORTS for row in df.collect()]
print(len(webi_ids))

In [0]:
user = "ESBWIL1"
password = "Built3@escape"

# # Dev Configurations
# bo_url = "https://bobidev.atradiusnet.com:8443/biprws/"
# base_path = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/dev"
# parameter_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_parameters"
# variable_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_variables"
# sql_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_metadata_cms"
# extract_summary_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.dev_webi_document_extraction_summary"

# Prod files
bo_url = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/"
base_path = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod"
parameter_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_parameters"
variable_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables"
datadiction_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary"
sql_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms"
excel_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_excel"
extract_summary_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary"


In [0]:
class webi_Document:
    def __init__(self, document_id):
        self.document_id = document_id
        self.document_cuid: str = None
        self.document_name: str = None
        self.folder_id: str = None
        self.full_path: str = None
        self.updated: str = None
        self.scheduled: str = None
        self.created: str = None
        self.lastAuthor: str = None
        self.refreshonopen: str = None
        self.data_providers: list = []
        self.schedules: list = []
        self.last_instance: "ScheduleInstance" = None
        self.size = None 
        self.binary_status: list = []  # Property to hold the status of pdf file and csv file
        self.variables: list = []
        self.parameters: list = []
        self.reports: int = None
        self.File_format: str=None
        self.File_location: str=None

    def set_doc_metadata(self, document_data):
        self.document_cuid = document_data.get("cuid")
        self.document_name = document_data.get("name")
        self.folder_id = document_data.get("folderId")
        self.full_path = document_data.get("path")
        self.updated = document_data.get("updated")
        self.scheduled = document_data.get("scheduled")
        self.created = document_data.get("createdBy")
        self.lastAuthor = document_data.get("lastAuthor")
        self.refreshonopen= document_data.get("refreshOnOpen")
        self.size = int(document_data.get("size")) if "size" in document_data else None

    def set_schedule_instances(self, schedule_instances):
        self.schedules = schedule_instances

    def add_provider(self, provider):
        self.data_providers.append(provider)

    def get_latest_schedule_id(self):
        if not self.schedules:
            return None
        latest_item = max(
            self.schedules,
            key=lambda s: datetime.fromisoformat(s["updated"].replace("Z", "+00:00")),
        )
        return latest_item["id"]

    def set_latest_schedule_instance(self, last_instance):
        self.last_instance = ScheduleInstance(last_instance)

    def persist_webi_metadata(
        self,
        spark,
        table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details",):
        if len(self.data_providers) == 0:
            raise ValueError("No Dataprovider infomation found for document")
        if self.last_instance is None:
            print(f"  Skipped persist_webi_metadata for {self.document_id}: scheduled=true but no schedule instance retrieved (API may have failed)")
            return None
        # print("\n")
        # print("persisting document and schedule instance")
        doc = self
        inst = doc.last_instance
        if inst.id is None:
            print(f"  Skipped persist_webi_metadata for {self.document_id}: schedule instance has no ID, cannot guarantee MERGE key uniqueness")
            return None
        row_data = {
            "document_id": str(doc.document_id),
            "document_cuid": doc.document_cuid,
            "document_name": doc.document_name,
            "folder_id": str(doc.folder_id) if doc.folder_id is not None else None,
            "full_path": doc.full_path,
            "document_updated_ts": doc.updated,
            "document_scheduled": doc.scheduled,
            "document_created_by": doc.created,
            "document_last_author": doc.lastAuthor,
            "schedule_id": str(inst.id) if inst.id is not None else None,
            "schedule_name": inst.name,
            "schedule_updated_ts": inst.updated,
            "schedule_format": inst.format,
            "schedule_status": inst.status,
            "repetition_type": inst.repetition_type,
            "repetition": json.dumps(inst.repetition),
            "destination_type": inst.destination_type,
            "destination": json.dumps(inst.destination),
            "parameters": json.dumps(inst.parameters),
            "ingestion_ts": datetime.utcnow(),
        }
        schema = StructType(
            [
                StructField("document_id", StringType()),
                StructField("document_cuid", StringType()),
                StructField("document_name", StringType()),
                StructField("folder_id", StringType()),
                StructField("full_path", StringType()),
                StructField("document_updated_ts", StringType()),  # cast to TimestampType via to_timestamp() below
                StructField("document_scheduled", StringType()),
                StructField("document_created_by", StringType()),
                StructField("document_last_author", StringType()),
                StructField("schedule_id", StringType()),
                StructField("schedule_name", StringType()),
                StructField("schedule_updated_ts", StringType()),  # cast to TimestampType via to_timestamp() below
                StructField("schedule_format", StringType()),
                StructField("schedule_status", StringType()),
                StructField("repetition_type", StringType()),
                StructField("repetition", StringType()),
                StructField("destination_type", StringType()),
                StructField("destination", StringType()),
                StructField("parameters", StringType()),
                StructField("ingestion_ts", TimestampType()),
            ]
        )
        df = spark.createDataFrame([row_data], schema=schema)
        df = df \
            .withColumn("document_updated_ts", to_timestamp(_col("document_updated_ts"))) \
            .withColumn("schedule_updated_ts", to_timestamp(_col("schedule_updated_ts")))
        print(f"Persisting data to {table_name} @ {datetime.now()}")
        df.write.format("delta").mode("append").saveAsTable(table_name)
        return None

    def persist_webi_excel(self, spark, table_name: str ="dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_excel"):
        webi_Excel_schema=StructType([
            StructField("Document_CUID", StringType()),
            StructField("Document_Id", StringType()),
            StructField("Document_name", StringType()),
            StructField("Folder_Id", StringType()),
            StructField("Full_path", StringType()),
            StructField("created", StringType()),
            StructField("lastAuthor", StringType()),
            StructField("scheduled", BooleanType()),
            StructField("updated", StringType()),
            StructField("Data_Provider_ID", StringType()),
            StructField("Data_Provider_Name", StringType()),
            StructField("DataSource_ID", StringType()),
            StructField("DataSource_CUID", StringType()),
            StructField("Data_Profider_Refresh_Time", StringType()),
            StructField("DataSource_Type", StringType()),
            StructField("DataSource_Name", StringType()),
            StructField("DataSource_Excel", BooleanType()),
            StructField("DataSource_Excel_Path", StringType()),
            StructField("DataSource_Excel_Tab", StringType()),
            StructField("ingestion_ts", TimestampType())
        ])
        if len(self.data_providers) == 0:
            raise ValueError("No Dataprovider infomation found for document")
        webi_dict: list =[]
        row_data = {
            "Document_Id": str(self.document_id),
            "Document_CUID": self.document_cuid,
            "Document_name": self.document_name,
            "Folder_Id": str(self.folder_id) if self.folder_id is not None else None,
            "Full_path": self.full_path,
            "updated": self.updated,
            "scheduled": str(self.scheduled).lower() == "true" if self.scheduled is not None else None,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "ingestion_ts": datetime.utcnow(),
        }
        for dataprovider in self.data_providers:
            if dataprovider.excel_used:
                provider_row_data = {
                    "Data_Provider_ID": str(dataprovider.id),
                    "Data_Provider_Name": dataprovider.name,
                    "Data_Profider_Refresh_Time": dataprovider.Dataprovider_refreshetime,
                    "DataSource_ID": dataprovider.datasource_id,
                    "DataSource_CUID": dataprovider.datasource_cuid,
                    "DataSource_Type": dataprovider.datasource_Type,
                    "DataSource_Name": dataprovider.datasource_Name,
                    "DataSource_Excel": dataprovider.excel_used,
                    "DataSource_Excel_Path": dataprovider.excel_path,
                    "DataSource_Excel_Tab": dataprovider.excel_tab,
                }
                combined_dict = {**row_data, **provider_row_data}
                webi_dict.append(combined_dict)
        df = spark.createDataFrame(webi_dict, schema=webi_Excel_schema)
        df.write.format("delta").mode("append").option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable(table_name)
        return None

    def persist_webi_datadictionary(self, spark, table_name: str ="dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary"):
        webi_dictionary_schema=StructType([
            StructField("Document_CUID", StringType()),
            StructField("Document_Id", StringType()),
            StructField("Document_name", StringType()),
            StructField("Folder_Id", StringType()),
            StructField("Full_path", StringType()),
            StructField("created", StringType()),
            StructField("lastAuthor", StringType()),
            StructField("scheduled", BooleanType()),
            StructField("updated", StringType()),
            StructField("Data_Provider_ID", StringType()),
            StructField("Data_Provider_Name", StringType()),
            StructField("DataSource_ID", StringType()),
            StructField("DataSource_CUID", StringType()),
            StructField("Data_Profider_Refresh_Time", StringType()),
            StructField("DataSource_Type", StringType()),
            StructField("DataSource_Name", StringType()),
            StructField("datafield_id", StringType()),
            StructField("datafield_name", StringType()),
            StructField("datafield_datatype", StringType()),
            StructField("datafield_qualification", StringType()),
            StructField("datafield_customSort", StringType()),
            StructField("datafield_stripped", StringType()),
            StructField("datafield_datasourceEnriched", StringType()),
            StructField("datafield_datasourceObjectId", StringType()),
            StructField("datafield_formulalanguageId", StringType()),
            StructField("datafield_datasourcePath", StringType()),
            StructField("ingestion_ts", TimestampType())
        ])
        if len(self.data_providers) == 0:
            raise ValueError("No Dataprovider infomation found for document")
        webi_dict: list =[]
        row_data = {
            "Document_Id": str(self.document_id),
            "Document_CUID": self.document_cuid,
            "Document_name": self.document_name,
            "Folder_Id": str(self.folder_id) if self.folder_id is not None else None,
            "Full_path": self.full_path,
            "updated": self.updated,
            "scheduled": str(self.scheduled).lower() == "true" if self.scheduled is not None else None,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "ingestion_ts": datetime.utcnow(),
        }
        for dataprovider in self.data_providers:
            provider_row_data = {
                "Data_Provider_ID": str(dataprovider.id),
                "Data_Provider_Name": dataprovider.name,
                "Data_Profider_Refresh_Time": dataprovider.Dataprovider_refreshetime,
                "DataSource_ID": dataprovider.datasource_id,
                "DataSource_CUID": dataprovider.datasource_cuid,
                "DataSource_Type": dataprovider.datasource_Type,
                "DataSource_Name": dataprovider.datasource_Name,
            }
            for datafield in dataprovider.dataDictionaries:
                datafield_row_data = {
                    "datafield_id": datafield.to_dict().get("datafield_id"),
                    "datafield_name": datafield.to_dict().get("datafield_name"),
                    "datafield_datatype": datafield.to_dict().get("datafield_datatype"),
                    "datafield_qualification": datafield.to_dict().get("datafield_qualification"),
                    "datafield_customSort": datafield.to_dict().get("datafield_customSort"),
                    "datafield_stripped": datafield.to_dict().get("datafield_stripped"),
                    "datafield_datasourceEnriched": datafield.to_dict().get("datafield_datasourceEnriched"),
                    "datafield_datasourceObjectId": datafield.to_dict().get("datafield_datasourceObjectId"),
                    "datafield_formulalanguageId": datafield.to_dict().get("datafield_formulalanguageId"),
                    "datafield_datasourcePath": datafield.to_dict().get("datafield_datasourcePath"),
                }
                combined_dict = {**row_data, **provider_row_data, **datafield_row_data}
                webi_dict.append(combined_dict)
        df = spark.createDataFrame(webi_dict, schema=webi_dictionary_schema)
        print(f"Persisting data to {table_name} @ {datetime.now()}")
        df.write.format("delta").mode("append").option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable(table_name)
        return None

    def persist_webi_sql(self, spark,
        table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_metadata_cms",):

        webi_metadata_cms_schema = StructType([
            StructField("Document_CUID", StringType()),
            StructField("Document_Id", StringType()),
            StructField("Document_name", StringType()),
            StructField("Folder_Id", StringType()),
            StructField("Full_path", StringType()),
            StructField("created", StringType()),
            StructField("lastAuthor", StringType()),
            StructField("scheduled", BooleanType()),
            StructField("updated", StringType()),
            StructField("API_Pause_Counts", StringType()),
            StructField("End_Time", StringType()),
            StructField("Execution_Time_in_seconds", StringType()),
            StructField("Extraction_Stats", StringType()),
            StructField("Number_of_API_Calls", StringType()),
            StructField("Number_of_Data_Providers", StringType()),
            StructField("File_format", StringType()),
            StructField("File_location", StringType()),
            StructField("Start_Time", StringType()),
            StructField("Data_Provider_ID", StringType()),
            StructField("Data_Provider_Name", StringType()),
            StructField("DataSource_ID", StringType()),
            StructField("DataSource_CUID", StringType()),
            StructField("Data_Profider_Refresh_Time", StringType()),
            StructField("DataSource_Type", StringType()),
            StructField("DataSource_Name", StringType()),
            StructField("SQL_Index", StringType()),
            StructField("SQL_Query", StringType()),
            StructField("Connection_ID", StringType()),
            StructField("Connection_Name", StringType()),
            StructField("Connection_Type", StringType()),
            StructField("Connection_DataBas", StringType()),
            StructField("Connection_Network", StringType()),
            StructField("ingestion_ts", TimestampType())
        ])
        # dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        if len(self.data_providers) == 0:
            raise ValueError("No Dataprovider infomation found for document")
        webi_sql: list =[]
        row_data = {
            "Document_Id": str(self.document_id),
            "Document_CUID": self.document_cuid,
            "Document_name": self.document_name,
            "Folder_Id": str(self.folder_id) if self.folder_id is not None else None,
            "Full_path": self.full_path,
            "updated": self.updated,
            "scheduled": str(self.scheduled).lower() == "true" if self.scheduled is not None else None,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "Number_of_Data_Providers": str(len(self.data_providers)),
            "File_format": self.File_format,
            "File_location": self.File_location,
            "ingestion_ts": datetime.utcnow(),
        }
        for dataprovider in self.data_providers:
            provider_row_data = {
                "Data_Provider_ID": str(dataprovider.id),
                "Data_Provider_Name": dataprovider.name,
                "Data_Profider_Refresh_Time": dataprovider.Dataprovider_refreshetime,
                "DataSource_ID": dataprovider.datasource_id,
                "DataSource_CUID": dataprovider.datasource_cuid,
                "DataSource_Type": dataprovider.datasource_Type,
                "DataSource_Name": dataprovider.datasource_Name,
            }
            for sql in dataprovider.sql_query:
                sql_row_data = {
                    "SQL_Index": sql.get("index"),
                    "SQL_Query": sql.get("sql_query"),
                }
                combined_dict = {**row_data, **provider_row_data, **sql_row_data}
                webi_sql.append(combined_dict)
        # display(pd.DataFrame(webi_sql))
        df = spark.createDataFrame(webi_sql, schema=webi_metadata_cms_schema)
        print(f"Persisting data to {table_name} @ {datetime.now()}")
        df.write.format("delta").mode("append").saveAsTable(table_name)
        return None
    
    
    def persist_webi_variable(self, spark,
        table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_variables",):

        variable_schema = StructType([
            StructField("Document_CUID", StringType()),
            StructField("Document_Id", StringType()),
            StructField("Document_name", StringType()),
            StructField("Folder_Id", StringType()),
            StructField("Full_path", StringType()),
            StructField("created", StringType()),
            StructField("lastAuthor", StringType()),
            StructField("scheduled", BooleanType()),
            StructField("updated", StringType()),

            StructField("variable_id", StringType()),
            StructField("variable_name", StringType()),
            StructField("variable_datatype", StringType()),
            StructField("variable_qualification", StringType()),
            StructField("variable_datasource_enriched", StringType()),
            StructField("variable_constant", StringType()),
            StructField("variable_formulalanguage_Id", StringType()),
            StructField("variable_definition", StringType()),
            StructField("ingestion_ts", TimestampType())
        ])
        row_data = {
            "Document_Id": str(self.document_id),
            "Document_CUID": self.document_cuid,
            "Document_name": self.document_name,
            "Folder_Id": str(self.folder_id) if self.folder_id is not None else None,
            "Full_path": self.full_path,
            "updated": self.updated,
            "scheduled": str(self.scheduled).lower() == "true" if self.scheduled is not None else None,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "ingestion_ts": datetime.utcnow(),
        }
        webi_variables = []
        for variable in self.variables:
            variable_row_data = variable.to_dict()
            combined_dict = {**row_data, **variable_row_data}
            webi_variables.append(combined_dict)
        df = spark.createDataFrame(webi_variables, schema=variable_schema)
        print(f"Persisting data to {table_name} @ {datetime.now()}")
        df.write.format("delta").mode("append").saveAsTable(table_name)
        return None
    

    def persist_webi_parameters(self, spark,
        table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_parameters",):
        parameter_class_schema = StructType([
            StructField("Document_CUID", StringType()),
            StructField("Document_Id", StringType()),
            StructField("Document_name", StringType()),
            StructField("Folder_Id", StringType()),
            StructField("Full_path", StringType()),
            StructField("created", StringType()),
            StructField("lastAuthor", StringType()),
            StructField("scheduled", BooleanType()),
            StructField("updated", StringType()),

            StructField("parameter_id", StringType()),
            StructField("parameter_name", StringType()),
            StructField("parameter_type", StringType()),
            StructField("parameter_dataprovider_Id", StringType()),
            StructField("parameter_optional", StringType()),
            StructField("parameter_dataprovider_link", StringType()),
            StructField("parameter_primary", StringType()),
            StructField("parameter_answer", StringType()),
            StructField("parameter_list_values", StringType()),
            StructField("ingestion_ts", TimestampType())
        ])

        row_data = {
            "Document_Id": str(self.document_id),
            "Document_CUID": self.document_cuid,
            "Document_name": self.document_name,
            "Folder_Id": str(self.folder_id) if self.folder_id is not None else None,
            "Full_path": self.full_path,
            "updated": self.updated,
            "scheduled": str(self.scheduled).lower() == "true" if self.scheduled is not None else None,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "ingestion_ts": datetime.utcnow(),
        }
        webi_parameters = []
        for parameter in self.parameters:
            parameter_row_data = parameter.to_dict()
            combined_dict = {**row_data, **parameter_row_data}
            webi_parameters.append(combined_dict)
        df = spark.createDataFrame(webi_parameters, schema=parameter_class_schema)
        print(f"Persisting Data to {table_name} @ {datetime.now()}")
        df.write.format("delta").mode("append").saveAsTable(table_name)
        return None
    
    def summary(self):
        return {
            "document_id": self.document_id,
            "document_cuid": self.document_cuid,
            "document_name": self.document_name,
            "full_path": self.full_path,
            "updated": self.updated,
            "scheduled": self.scheduled,
            "created": self.created,
            "lastAuthor": self.lastAuthor,
            "count_reports":self.reports,
            "variables_count": len(self.variables) if self.variables is not None else 0,
            "data_fields_count":sum(len(dp.dataDictionaries) for dp in self.data_providers) if self.data_providers is not None else 0,
            # len(self.dataDictionaries) if self.dataDictionaries is not None else 0,
            "parameters": len(self.parameters) if self.parameters is not None else 0,
            "sql_queries": sum(len(dp.sql_query) for dp in self.data_providers) if self.data_providers is not None else 0,
            "binary_status": json.dumps(self.binary_status if self.binary_status is not None else []),
            "status": "Document Retrieved"
        }


In [0]:
class ScheduleInstance:
    def __init__(self, schedule_instance_data):
        self.id = schedule_instance_data.get("id")
        self.name = schedule_instance_data.get("name")
        self.updated = schedule_instance_data.get("updated")
        self.format = schedule_instance_data.get("format", {}).get("@type")
        self.status = schedule_instance_data.get("status", {}).get("$")
        self.parameters = schedule_instance_data.get("parameters", {})
        self.repetition_type = None
        self.repetition = {}
        self.destination_type = None
        self.destination = {}
        self.get_repetition(schedule_instance_data)
        self.get_destination(schedule_instance_data)

    def get_repetition(self, schedule_instance_data):
        SCHEDULE_KEYS = {"once", "hourly", "daily", "weekly", "monthly", "calendar"}
        completed = False
        for key in SCHEDULE_KEYS:
            if key in schedule_instance_data:
                self.repetition_type = key
                self.repetition = schedule_instance_data[key]
                completed = True
                return None
        if not completed:
            print("No repetition type found in the schedule instance data.")
        return None

    def get_destination(self, schedule_instance_data):
        DESTINATION_KEYS = {"mail", "inbox", "file", "ftp"}
        completed = False
        destination_block = schedule_instance_data.get("destination", {})
        for key in DESTINATION_KEYS:
            if key in destination_block:
                self.destination_type = key
                self.destination = destination_block[key]
                completed = True
                return None
        if not completed:
            print("No destination type found in the schedule instance data.")
        return None

    def __repr__(self):
        return (
            f"ScheduleInstance(,\n"
            f"id={self.id}, \n"
            f"name={self.name}, \n"
            f"updated={self.updated}, \n"
            f"format is {self.format}, \n"
            f"with status {self.status}, \n"
            f"repetition_type={self.repetition_type}, \n"
            f"destination_type={self.destination_type}"
            f")"
        )

In [0]:

class Dataprovider:
    def __init__(self, data_provider: dict):
        self.id = data_provider.get("id")
        self.name = data_provider.get("name")
        self.datasource_id = data_provider.get("dataSourceId")
        self.datasource_cuid = None
        self.datasource_Type = data_provider.get("dataSourceType")
        self.datasource_Name = None
        self.connection_id = None
        self.connection_name = None
        self.connection_Type = None
        self.connection_Database = None
        self.connection_Network = None
        self.sql_query:list = []
        self.Dataprovider_refreshetime = None
        self.Dataprovider_rowcount= int(data_provider.get("rowCount")) if "rowCount" in data_provider else None
        self.dataDictionaries: list = [] # property for all available data fields
        self.excel_used: bool = None
        self.excel_path: str = None
        self.excel_tab: str = None


    def set_DP_Details(self, data_provider_details: dict):
        self.datasource_cuid = data_provider_details.get("dataSourceCuid")
        self.datasource_Name = data_provider_details.get("dataSourceName")
        self.Dataprovider_refreshetime = data_provider_details.get("updated")

    def append_sql(self, index: str, sql: str):
        if self.sql_query is None:
            self.sql_query = []
        self.sql_query.append({"index": index, "sql_query": sql})
        # [index] = sql

In [0]:
class dataDiction: 
    def __init__(self, data_dict: dict):
        self.id=data_dict.get("id") if "id" in data_dict else None
        self.name = data_dict.get("name") if "name" in data_dict else None
        self.dataType = data_dict.get("@dataType") if "@dataType" in data_dict else None
        self.qualification = data_dict.get("@qualification") if "@qualification" in data_dict else None
        self.customSort = data_dict.get("@customSort") if "@customSort" in data_dict else None
        self.stripped = data_dict.get("@stripped") if "@stripped" in data_dict else None
        self.dataSourceEnriched = data_dict.get("@dataSourceEnriched") if "@dataSourceEnriched" in data_dict else None
        self.dataSourceObjectId = data_dict.get("dataSourceObjectId") if "dataSourceObjectId" in data_dict else None
        self.formulaLanguageId = data_dict.get("formulaLanguageId") if "formulaLanguageId" in data_dict else None
        self.dataSourcePath = data_dict.get("dataSourcePath",{}) if "dataSourcePath" in data_dict else None
    def to_dict(self):
        return {
            "datafield_id": self.id,
            "datafield_name": self.name,
            "datafield_datatype": self.dataType,
            "datafield_qualification": self.qualification,
            "datafield_customSort": self.customSort,
            "datafield_stripped": self.stripped,
            "datafield_datasourceEnriched": self.dataSourceEnriched,
            "datafield_datasourceObjectId": self.dataSourceObjectId,
            "datafield_formulalanguageId": self.formulaLanguageId,
            "datafield_datasourcePath": json.dumps(self.dataSourcePath)
        }

In [0]:
class Variable:
    def __init__(self, variable_data: dict):
        self.id = variable_data.get("id")
        self.name = variable_data.get("name")
        self.datatype = variable_data.get("@dataType") if "@dataType" in variable_data else None
        self.qualification = variable_data.get("@qualification") if "@qualification" in variable_data else None
        self.datasource_enriched = variable_data.get("@dataSourceEnriched") if "@dataSourceEnriched" in variable_data else None
        self.constant = variable_data.get("@constant") if "@constant" in variable_data else None
        self.formulalanguage_Id = None
        self.definition = None

    def enrich_variables(self, variable_details:dict):
        self.formulalanguage_Id = variable_details.get("formulaLanguageId") if "formulaLanguageId" in variable_details else None
        self.definition = variable_details.get("definition") if "definition" in variable_details else None

    def to_dict(self):
        return {
            "variable_id": self.id,
            "variable_name": self.name,
            "variable_datatype": self.datatype,
            "variable_qualification": self.qualification,
            "variable_datasource_enriched": self.datasource_enriched,
            "variable_constant": self.constant,
            "variable_formulalanguage_Id": self.formulalanguage_Id,
            "variable_definition": self.definition
        }


In [0]:
class parameter_class:
    def __init__(self, parameter_data: dict):
        self.id = parameter_data.get("id")
        self.name = parameter_data.get("name")
        self.type = parameter_data.get("@type") if "@type" in parameter_data else None
        self.dataprovider_Id = parameter_data.get("@dpId") if "@dpId" in parameter_data else None
        self.optional = parameter_data.get("@optional") if "@optional" in parameter_data else None
        self.dataprovider_link = parameter_data.get("@dpLinks") if "@dpLinks" in parameter_data else None
        self.primary = parameter_data.get("@primary") if "@primary" in parameter_data else None
        self.answer = parameter_data.get("answer",{}) if "answer" in parameter_data else None
        self.list_values = []
        if isinstance(self.answer, dict) and "info" in self.answer:
            info = self.answer["info"]
            if isinstance(info, dict) and "lov" in info:
                lov = info["lov"]
                if isinstance(lov, dict) and "values" in lov:
                    values = lov["values"]
                    if isinstance(values, dict) and "value" in values:
                        self.list_values = values["value"]
    
    def to_dict(self):
        return {
            "parameter_id": self.id,
            "parameter_name": self.name,
            "parameter_type": self.type,
            "parameter_dataprovider_Id": self.dataprovider_Id,
            "parameter_optional": self.optional,
            "parameter_dataprovider_link": self.dataprovider_link,
            "parameter_primary": self.primary,
            "parameter_answer": json.dumps(self.answer),
            "parameter_list_values": " | ".join(map(str, self.list_values))
        }


In [0]:
class SAP_BO_Connection:
    def __init__(self, bo_url, user, password, retrial:int= 3):
        self.user = user
        self.password = password
        self.bo_url = bo_url
        self.logon_status = False
        self.retrial=retrial
        self.conection_timeout=5
        self.sleeptime=5
        self.logonToken = self.logon_request()
        self.headers = {
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }
        self.pdf_headers = {
            "Content-Type": "application/pdf",
            "Accept": "application/pdf",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }
        self.html_headers = {
            "Content-Type": "text/html",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }

    def logon_request(self):
        logon_url = f"{self.bo_url}/logon/long"
        auth_type = "secLDAP"
        logon_payload = {
            "userName": self.user,
            "password": self.password,
            "auth": auth_type,
        }
        headers = {"Content-Type": "application/json"}
        try:
            response = requests.post(
                logon_url, json=logon_payload, headers=headers, verify=False
            )
            logon_token = response.headers.get("X-SAP-LogonToken", None)
            if logon_token is not None:
                print(datetime.now(), "Logon successful.")
                print("Logon Token:", logon_token)
                self.logon_status = True  
            else: 
                self.logon_status = False
        except Exception as e:
            print(
                "Failed to logon SAP BO Server:",
                logon_url,
                " with error: ",
                e,
                "exiting script.",
            )
            self.logon_status = False
            logon_token = None
        return logon_token

    def logoff_request(self):
        if self.logon_status is True:
            print("Logging off from BO REST API.")
            logoff_url = f"{self.bo_url}/logoff"
            logoff_headers = {"X-SAP-LogonToken": self.logonToken}
            logoff_response=requests.post(logoff_url, headers=logoff_headers, verify=False)
            print(f"Logged off successfully: {logoff_response.status_code}  ")
            self.logon_status = False
            return True
        else:
            raise Exception("Not a valid logon token")
            return False

    def get_providers(self, document_id):
        DP_Api_url = (
            f"{self.bo_url}raylight/v1/documents/{str(document_id)}/dataproviders"
        )
        if self.logon_status is True:
            try:
                i=1
                document_response = None
                while i<=self.retrial and (document_response is None or document_response.status_code!=200):
                    document_response = requests.get(
                    DP_Api_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False)
                    i+=1
                    if document_response.status_code==200:
                        document_data = document_response.json()
                    elif document_response.status_code==401:
                        raise Exception(document_response.json().get("message"))
                    else:
                        time.sleep(self.sleeptime)
                return document_data
            except Exception as e:
                print(
                    "Failed to retrieve providers data:",
                    DP_Api_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
                return None
        else:
            raise Exception("Not a valid logon token")
            return None

    def get_document(self, document_id):
        if self.logon_status is True:
            document_url = f"{self.bo_url}raylight/v1/documents/{document_id}"
            try:
                i=1
                document_response = None
                while i<=self.retrial and (document_response is None or document_response.status_code!=200):
                    document_response = requests.get(
                        document_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i+=1
                    if document_response.status_code==200:
                        document_data = document_response.json()
                    elif document_response.status_code==401:
                        raise Exception(document_response.json().get("message"))
                    elif document_response.status_code==404:
                        raise Exception(document_response.json().get("message"))
                    else:
                        time.sleep(self.sleeptime)
                
                return document_data
            except Exception as e:
                print(
                    "Failed to retrieve document data:",
                    document_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")
            return None

    def get_document_pdf(self, doc: webi_Document, base_path: str):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        document_url = f"{self.bo_url}raylight/v1/documents/{doc.document_id}?format=pdf&refresh=false"
        
        # Build target path
        ingestion_date = datetime.now().strftime("%Y-%m-%d")
        file_name = f"WEBI_{doc.document_id}.pdf"
        target_dir = f"{base_path}/object_type=pdf/ingestion_date={ingestion_date}"
        os.makedirs(target_dir, exist_ok=True)
        target_path = f"{target_dir}/{file_name}"
        if doc.size is not None and doc.size > 5242880:
            read_timeout = 120 * doc.size / 5242880
        elif doc.data_providers and any(dp.Dataprovider_rowcount for dp in doc.data_providers):
            total_rowcount = sum(dp.Dataprovider_rowcount for dp in doc.data_providers if dp.Dataprovider_rowcount)
            if total_rowcount < 50000:
                total_rowcount = 50000
            read_timeout = 120 * total_rowcount / 50000
        else:
            read_timeout = 120
        try:
            i=1
            response = None
            while i <= self.retrial and (response is None or response.status_code != 200):
                read_timeout = min(900, read_timeout + (i-1)*120)
                print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                print(f"Reading timeout set to {int(read_timeout)} seconds for downloading pdf file")
                i += 1
                with requests.get(
                    document_url,
                    headers=self.pdf_headers,
                    stream=True,
                    timeout=(self.conection_timeout, read_timeout),
                    verify=False,
                ) as response:
                    response.raise_for_status()
                    with open(target_path, "wb") as f:
                        for chunk in response.iter_content(chunk_size=8192 * 4):
                            if chunk:
                                f.write(chunk)
                if response.status_code==401:
                    raise Exception(response.json().get("message"))
                elif response.status_code != 200:
                    time.sleep(self.sleeptime)
                print(f"Finished downloading file at time: {datetime.now()}")

            return {
                "document_id": doc.document_id,
                "file_format": "pdf",
                "file_path": target_path,
                "status": "SUCCESS"
            }
        except Exception as e:
            print(
                "Failed to retrieve document_pdf data:",
                document_url,
                "with error:",
                e,
                "exiting call.",
            )
            raise Exception(e)

    def get_document_csv(self, doc: webi_Document, base_path: str):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        document_url = f"{self.bo_url}raylight/v1/documents/{doc.document_id}?format=html&refresh=false"
        # Build target path
        ingestion_date = datetime.now().strftime("%Y-%m-%d")
        file_name = f"WEBI_{doc.document_id}.csv"
        target_dir = f"{base_path}/object_type=csv/ingestion_date={ingestion_date}"
        os.makedirs(target_dir, exist_ok=True)
        target_path = f"{target_dir}/{file_name}"
        if doc.size is not None and doc.size > 5242880:
            read_timeout = 120 * doc.size / 5242880
        elif doc.data_providers and any(dp.Dataprovider_rowcount for dp in doc.data_providers):
            total_rowcount = sum(dp.Dataprovider_rowcount for dp in doc.data_providers if dp.Dataprovider_rowcount)
            if total_rowcount < 50000:
                total_rowcount = 50000
            read_timeout = 120 * total_rowcount / 50000
        else:
            read_timeout = 120
        try:
            i=1
            response = None
            while i <= self.retrial and (response is None or response.status_code != 200):
                read_timeout = min(900, read_timeout + (i-1)*120)
                print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                print(f"Reading timeout set to {int(read_timeout)} seconds for downloading csv file")
                i += 1
                with requests.get(
                    document_url,
                    headers=self.html_headers,
                    stream=True,
                    timeout=(self.conection_timeout, read_timeout),
                    verify=False,
                ) as response:
                    response.raise_for_status()
                    with open(target_path, "wb") as f:
                        for chunk in response.iter_content(chunk_size=8192 * 4):
                            if chunk:
                                f.write(chunk)
                print(f"Finished downloading file at time: {datetime.now()}")
                if response.status_code==401:
                    raise Exception(response.json().get("message"))
                elif response.status_code != 200:
                    time.sleep(self.sleeptime)
            return {
                "document_id": doc.document_id,
                "file_format": "csv",
                "file_path": target_path,
                "status": "SUCCESS"
            }
        except Exception as e:
            print(
                "Failed to retrieve document_csv data:",
                document_url,
                "with error:",
                e,
                "exiting call.",
            )
            raise Exception(e)
    
    def get_provider_details(self, document_id, provider_id):
        if self.logon_status is True:
            provider_url = f"{self.bo_url}raylight/v1/documents/{document_id}/dataproviders/{provider_id}"
            try:
                i=1
                document_response = None
                while i <= self.retrial and (document_response is None or document_response.status_code != 200):
                    document_response = requests.get(
                        provider_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i += 1
                    document_data = document_response.json()
                    if document_response.status_code==401:
                        raise Exception(document_response.json().get("message"))
                    elif document_response.status_code != 200:
                        time.sleep(self.sleeptime)
                return document_data
            except Exception as e:
                print(
                    "Failed to retrieve provider_details data:",
                    provider_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")
            return None

    def get_provider_SQL(self, document_id, provider_id):
        if self.logon_status is True:
            provider_url = f"{self.bo_url}raylight/v1/documents/{document_id}/dataproviders/{provider_id}/queryplan"
            try:
                i=1
                document_response = None
                while i <= self.retrial and (document_response is None or document_response.status_code != 200):
                    document_response = requests.get(
                        provider_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i += 1
                    document_data = document_response.json()
                    if document_response.status_code==401:
                        raise Exception(document_response.json().get("message"))
                    elif document_response.status_code != 200:
                        time.sleep(self.sleeptime)
                return document_data
            except Exception as e:
                print(
                    "Failed to retrieve SQL data:",
                    provider_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")
        return None
    def get_schedules(self, document_id):
        if self.logon_status is True:
            schedules_url = (
                f"{self.bo_url}raylight/v1/documents/{document_id}/schedules"
            )
            try:
                i=1
                Schedules_response = None
                while i <= self.retrial and (Schedules_response is None or Schedules_response.status_code != 200):

                    Schedules_response = requests.get(
                        schedules_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i += 1
                    schedules_data = Schedules_response.json()
                    if Schedules_response.status_code==401:
                        raise Exception(Schedules_response.json().get("message"))
                    elif Schedules_response.status_code != 200:
                        time.sleep(self.sleeptime)
                return schedules_data
            except Exception as e:
                print(
                    "Failed to retrieve schedules data:",
                    schedules_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")

    def get_schedule_details(self, document_id, schdule_id):
        if self.logon_status is True:
            instance_url = f"{self.bo_url}raylight/v1/documents/{document_id}/schedules/{schdule_id}"
            try:
                i=1
                Instance_response = None
                while i <= self.retrial and (Instance_response is None or Instance_response.status_code != 200):
                    Instance_response = requests.get(
                        instance_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i += 1
                    schedules_data = Instance_response.json()
                    if Instance_response.status_code==401:
                        raise Exception(Instance_response.json().get("message"))
                    elif Instance_response.status_code != 200:
                        time.sleep(self.sleeptime)
                return schedules_data
            except Exception as e:
                print(
                    "Failed to retrieve schedule_details data:",
                    instance_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")

    def get_document_variables(self, doc: webi_Document):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        variable_url = f"{self.bo_url}raylight/v1/documents/{doc.document_id}/variables"
        try:
            i=1
            variable_response = None
            while i <= self.retrial and (variable_response is None or variable_response.status_code != 200):

                variable_response = requests.get(
                    variable_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                )
                i += 1
                variable_data = variable_response.json()
                if variable_response.status_code==401:
                    raise Exception(variable_response.json().get("message"))
                elif variable_response.status_code != 200:
                    time.sleep(self.sleeptime)
            return variable_data
        except Exception as e:
            print(
                "Failed to retrieve variables data:",
                variable_url,
                "with error:",
                e,
                "exiting call.",
            )
            raise Exception(e)

    def get_variable_details(self, doc: webi_Document, variable_id:str):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        variable_url = f"{self.bo_url}raylight/v1/documents/{doc.document_id}/variables/{variable_id}"
        try:
            i=1
            variable_response = None
            while i <= self.retrial and (variable_response is None or variable_response.status_code != 200):

                variable_response = requests.get(
                    variable_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                )
                i += 1
                variable_data = variable_response.json()
                if variable_response.status_code==401:
                    raise Exception(variable_response.json().get("message"))
                elif variable_response.status_code != 200:
                    time.sleep(self.sleeptime)
            return variable_data
        except Exception as e:
            print(
                "Failed to retrieve variable_details data:",
                variable_url,
                "with error:",
                e,
                "exiting call.",
            )
            raise Exception(e)
    
    def get_parameters(self, webi_id: str):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        parameters_url = f"{self.bo_url}raylight/v1/documents/{webi_id}/parameters"
        try:
            i=1
            parameters_response = None
            while i <= self.retrial and (parameters_response is None or parameters_response.status_code != 200):

                parameters_response = requests.get(
                    parameters_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                )
                i += 1
                parameters_data = parameters_response.json()
                if parameters_response.status_code==401:
                    raise Exception(parameters_response.json().get("message"))
                elif parameters_response.status_code != 200:
                    time.sleep(self.sleeptime)
            return parameters_data
        except Exception as e:
            print(
                "Failed to retrieve parameters data:",
                parameters_url,
                "with error:",
                e,
                "exiting call.",
            )
            raise Exception(e)
    
    def get_reports(self, webi_id: str):
        if not self.logon_status:
            raise Exception("Not a valid logon token")
            return None
        reports_url = f"{self.bo_url}raylight/v1/documents/{webi_id}/reports"
        try:
            i=1
            reports_response = None
            while i <= self.retrial and (reports_response is None or reports_response.status_code != 200):

                reports_response = requests.get(
                    reports_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                )
                i += 1
                if reports_response is not None and reports_response.status_code == 200:
                    json_data = reports_response.json()
                    reports_data = json_data.get("reports",{})if "reports" in json_data else None
                    report_data = reports_data.get("report", []) if "report" in reports_data else None
                elif reports_response.status_code==401:
                    raise Exception(reports_response.json().get("message"))
                elif reports_response.status_code != 200:
                    time.sleep(self.sleeptime)
                    reports_data = None
                
            return report_data
        except Exception as e:
            print(
                "Failed to retrieve reports data:",
                reports_url,
                "with error:",
                e,
                "exiting call.",)
            raise Exception(e)


In [0]:
def process_document(webi_id: str, bo_connection: SAP_BO_Connection):
    try:
        Document_data = bo_connection.get_document(webi_id)
    except Exception as e:
        # print(f"  Failed to retrieve document {webi_id}: {e}")
        raise Exception(e)
    if Document_data is None:
        print(f"  Failed to retrieve document {webi_id}")
        return webi_Document(webi_id)
    WEBI_doc = webi_Document(webi_id)
    WEBI_doc.set_doc_metadata(Document_data["document"])
    # reportdata=bo_connection.get_reports(webi_id)
    # WEBI_doc.reports = len(reportdata) if reportdata is not None else 0
    try:
        data_providers_response = bo_connection.get_providers(webi_id)
    except Exception as e:
        if "Not a valid logon token" in str(e):
            raise
        print(f"  Failed to retrieve data providers for document {webi_id}: {e}")
        data_providers_response = {}
    if data_providers_response is None:
        data_providers_response = {}
    data_providers = data_providers_response.get("dataproviders", {}).get(
        "dataprovider", []
    )
    if isinstance(data_providers, dict):
        data_providers = [data_providers]
    for data_provider in data_providers:
        webi_provider = Dataprovider(data_provider)
        try:
            provider_details = bo_connection.get_provider_details(webi_id, webi_provider.id)
        except Exception as e:
            if "Not a valid logon token" in str(e):
                raise
            print(f"  Failed to retrieve provider details for provider {webi_provider.id} in document {webi_id}: {e}")
            provider_details = {}
        if isinstance(provider_details.get("dataprovider", {}), dict):
            webi_provider.set_DP_Details(provider_details.get("dataprovider", {}))
            if isinstance(provider_details.get("dataprovider", {}).get("dictionary",{}).get("expression", []), list):
                for dataDict in provider_details.get("dataprovider", {}).get("dictionary",{}).get("expression", []):
                    webi_provider.dataDictionaries.append(dataDiction(dataDict))
            if provider_details.get("dataprovider", {}).get("dataSourceType") == "fhsql":
                properties = provider_details.get("dataprovider", {}).get("properties", {}).get("property",[])
                # SAP BO API returns a dict when there is exactly 1 property, and a list when there are multiple
                if isinstance(properties, dict):
                    properties = [properties]
                for prop in properties:
                    if prop.get("@key") == "sql":
                        sql_query = prop.get("$", "")
                        Row_index="1"
                        webi_provider.append_sql(Row_index, sql_query)
            elif provider_details.get("dataprovider", {}).get("dataSourceType") == "excel":
                webi_provider.excel_used = True
                webi_provider.excel_path = provider_details.get("dataprovider", {}).get("dataSourceLongName")
                webi_provider.excel_tab = provider_details.get("dataprovider", {}).get("name")
                webi_provider.append_sql("0", "Data Source Type excel not handled for SQL extraction")
            else:
                try:
                    Query_response = bo_connection.get_provider_SQL(webi_id, webi_provider.id)
                except Exception as e:
                    if "Not a valid logon token" in str(e):
                        raise
                    print(f"  Failed to retrieve provider SQL for provider {webi_provider.id} in document {webi_id}: {e}")
                    Query_response = {}
                Query = Query_response.get("queryplan", {}) if Query_response else {}
                if len(Query) != 0:
                    List_Plans = loop_Dict(Query)
                    Dataframe = pd.json_normalize(List_Plans)
                    for _, row in Dataframe.iterrows():
                        sql_query = row.get("sql_query", "")
                        Row_index = row.get("index", "")
                        webi_provider.append_sql(Row_index, sql_query)
                else:
                    sql_query = "Error retrieving Query Plan"
                    Row_index = "1"
                    webi_provider.append_sql(Row_index, sql_query)
        WEBI_doc.data_providers.append(webi_provider)
    print(f"{len(WEBI_doc.data_providers)} data provider(s) appended")
    # schedules not included here in controller extraction list
    if WEBI_doc.scheduled == "true":
        print("Document is scheduled")
        try:
            schedule_package = bo_connection.get_schedules(WEBI_doc.document_id)
        except Exception as e:
            if "Not a valid logon token" in str(e):
                raise
            print(f"  Failed to retrieve schedules for document {WEBI_doc.document_id}: {e}")
            schedule_package = {}
        if schedule_package and "schedules" in schedule_package and "schedule" in schedule_package["schedules"]:
            WEBI_doc.set_schedule_instances(schedule_package["schedules"]["schedule"])
            try:
                last_instance_data = bo_connection.get_schedule_details(
                    WEBI_doc.document_id, WEBI_doc.get_latest_schedule_id()
                )
            except Exception as e:
                if "Not a valid logon token" in str(e):
                    raise
                print(f"  Failed to retrieve schedule details for document {WEBI_doc.document_id}: {e}")
                last_instance_data = {}
            if last_instance_data and "schedule" in last_instance_data:
                WEBI_doc.set_latest_schedule_instance(last_instance_data["schedule"])
        else:
            print("  No schedules found for document")
    try:
        Variables = bo_connection.get_document_variables(WEBI_doc)
    except Exception as e:
        if "Not a valid logon token" in str(e):
            raise
        print(f"  Failed to retrieve variables for document {WEBI_doc.document_id}: {e}")
        Variables = {}
    if Variables is None:
        Variables = {}
    for variable in Variables.get("variables",{}).get("variable",[]) :
        Variable_ind = Variable(variable)
        try:
            variable_definition = bo_connection.get_variable_details(WEBI_doc, Variable_ind.id)
        except Exception as e:
            if "Not a valid logon token" in str(e):
                raise
            print(f"  Failed to retrieve variable details for variable {Variable_ind.id} in document {WEBI_doc.document_id}: {e}")
            variable_definition = {}
        if variable_definition is None:
            variable_definition = {}
        Variable_ind.enrich_variables(variable_definition.get("variable",{}))
        WEBI_doc.variables.append(Variable_ind)
    try:
        parameters = bo_connection.get_parameters(WEBI_doc.document_id)
    except Exception as e:
        if "Not a valid logon token" in str(e):
            raise
        print(f"  Failed to retrieve parameters for document {WEBI_doc.document_id}: {e}")
        parameters = {}
    if parameters is None:
        parameters = {}
    for parameter in parameters.get("parameters",{}).get("parameter",[]) :
        Parameter_ind = parameter_class(parameter)
        WEBI_doc.parameters.append(Parameter_ind)
    return WEBI_doc

In [0]:
# Functions to dynamically loop through the JSON response and extract the SQL queries
def loop_Dict(Raw_data) -> list:
    Output = []
    index = ""
    sql_query = ""
    for key, values in Raw_data.items():
        if isinstance(values, dict):
            Output.extend(loop_Dict(values))
        elif isinstance(values, list):
            Output.extend(loop_List(values))
        else:
            if key == "@index":
                index = values
            elif key == "$":
                sql_query = values
    if index != "" and sql_query != "":
        Output.append({"index": index, "sql_query": sql_query})
    return Output


def loop_List(Raw_data) -> list:
    Output = []
    index = ""
    sql_query = ""
    for prop in Raw_data:
        if type(prop) is dict:
            if prop.get("statement", []):
                Output.extend(loop_List(prop.get("statement", [])))
            elif prop.get("statement", {}):
                Output.extend(loop_List(prop.get("statement", {})))
            else:
                Output.extend(loop_List(prop))
            continue
        if prop == "@index":
            index = Raw_data[prop]
        elif prop == "$":
            sql_query = Raw_data[prop]
    if index != "" and sql_query != "":
        Output.append({"index": index, "sql_query": sql_query})
    return Output

In [0]:
try:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    bo_connection = SAP_BO_Connection(bo_url, user, password)
    if not bo_connection.logon_status:
        print("Failed to logon. Exiting.")
        raise SystemExit("Logon failed")

    # Load target summary schema once to avoid repeated table reads per iteration
    extract_summary_schema = spark.table(extract_summary_table).schema
    documents_summary = []
    last_logon_time = datetime.now()
    REAUTH_INTERVAL_SEC = 45 * 60  # re-authenticate every 45 minutes
    for i, webi_id in enumerate(webi_ids):
        print(f"\n--- Processing document {i+1}/{len(webi_ids)}: ID={webi_id} ---")
        # --- Periodic re-authentication ---
        if (datetime.now() - last_logon_time).total_seconds() >= REAUTH_INTERVAL_SEC:
            print(f"\n*** 45-min token refresh at {datetime.now().strftime('%H:%M:%S')} — re-authenticating ***")
            try:
                bo_connection.logoff_request()
            except Exception as _reauth_e:
                print(f"  Logoff during re-auth failed (ignored): {_reauth_e}")
            bo_connection = SAP_BO_Connection(bo_url, user, password)
            if not bo_connection.logon_status:
                print("Failed to re-logon. Exiting.")
                raise SystemExit("Re-logon failed after 45-min refresh")
            last_logon_time = datetime.now()
            print(f"*** Re-logon successful at {last_logon_time.strftime('%H:%M:%S')} ***")
        doc_summary = {}
        try:
            webi = process_document(str(webi_id), bo_connection)
        except Exception as e:
            print(f"Document level error: {e}")
            if "Not a valid logon token" in str(e):
                bo_connection.logoff_request()
                bo_connection = SAP_BO_Connection(bo_url, user, password)
                if not bo_connection.logon_status:
                    print("Failed to logon. Exiting.")
                    raise SystemExit("Logon failed")
                else:
                    print("\n********Retry Login*********")
                    last_logon_time = datetime.now()
                    webi = process_document(str(webi_id), bo_connection)
            elif "does not exist" in str(e):
                webi = webi_Document(webi_id)
                webi.document_name = "Does not exist"
                doc_summary = {"document_id": webi_id, "document_name": "Does not exist", "status": "SKIP_NO_DATA","ingestion_ts": datetime.utcnow()}
                if doc_summary:
                    aligned = {f.name: doc_summary.get(f.name) for f in extract_summary_schema.fields}
                    spark.createDataFrame([aligned], schema=extract_summary_schema).write.mode("append").saveAsTable(extract_summary_table)
        try:
            if webi.document_name is None:
                print(f"{webi_id} has Document Name: {webi.document_name}\n  Skipped: no valid document data")
                documents_summary.append({"document_id": webi_id, "status": "SKIP_NO_DATA","ingestion_ts": datetime.utcnow()})
            elif webi.document_name != 'Does not exist':
                print(f"Document Name in {webi.document_name}")
                try:
                    webi.persist_webi_datadictionary(spark=spark, table_name=datadiction_table)
                    webi.persist_webi_variable(spark=spark, table_name=variable_table)
                    webi.persist_webi_parameters(spark=spark, table_name=parameter_table)
                    webi.persist_webi_excel(spark=spark, table_name=excel_table)
                    if webi.scheduled == "true":
                        webi.persist_webi_metadata(spark=spark)
                except Exception as e:
                    print(f"Document level error: {e}")
                try:
                    if str(webi.refreshonopen).lower() == "false" or any(dp.Dataprovider_rowcount and dp.Dataprovider_rowcount > 0 for dp in webi.data_providers):
                        # # Review with Pablo voted for CSV reports data, no PDF required.
                        # result_pdf = bo_connection.get_document_pdf(webi, base_path)
                        # webi.binary_status.append(result_pdf if result_pdf is not None else {'document_id': str(webi_id), 'file_format': 'pdf', 'file_path': None, 'status': 'FAIL'})
                        result_csv = bo_connection.get_document_csv(webi, base_path)
                        if result_csv is not None:
                            webi.File_format=result_csv['file_format']
                            webi.File_location = result_csv['file_path']
                        else: 
                            webi.File_location = 'API CSV extraction Failed'
                        webi.binary_status.append(result_csv if result_csv is not None else {'document_id': str(webi_id), 'file_format': 'csv', 'file_path': None, 'status': 'FAIL'})
                    else:
                        print(f"  Skipped: refreshOnOpen={webi.refreshonopen} (risk refreshing)")
                        webi.binary_status.append({"document_id": str(webi_id), "status": "SKIP to avoid Data refreshing"})
                finally:
                    webi.persist_webi_sql(spark=spark, table_name=sql_table)
                    doc_summary = webi.summary()
                    doc_summary["ingestion_ts"] = datetime.utcnow()
                    if doc_summary:
                        aligned = {f.name: doc_summary.get(f.name) for f in extract_summary_schema.fields}
                        spark.createDataFrame([aligned], schema=extract_summary_schema).write.mode("append").saveAsTable(extract_summary_table)

        except Exception as e:
            print(f"  ERROR processing {webi_id}: {e}")
            if doc_summary:
                aligned = {f.name: doc_summary.get(f.name) for f in extract_summary_schema.fields}
                spark.createDataFrame([aligned], schema=extract_summary_schema).write.mode("append").saveAsTable(extract_summary_table)

finally:
    print(f"\n\n=== BATCH COMPLETE: Processed {len(webi_ids)} documents ===")
    bo_connection.logoff_request()
        # print("Logoff successful")


In [0]:
import requests as req
import json
from datetime import datetime
from databricks.sdk import WorkspaceClient
import pytz

# Only self-trigger if within the allowed time window (weekdays 11:30 - 23:00 CET)
# Use notebook context (reliable in both serverless and existing-cluster job runs)
ctx = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson())
job_id = ctx.get("tags", {}).get("jobId")

w = WorkspaceClient()
print(w.current_user.me())

if job_id:
    cet = pytz.timezone("Europe/Amsterdam")
    now_cet = datetime.now(cet)
    cutoff_hour = 21  # Stop at 23:00 CET
    start_hour = 12  # Start at 11:30 CET
    is_weekday = now_cet.weekday() < 5  # Mon=0, Fri=4

    print( f"Start of day checking: {now_cet.hour < start_hour}" )
    print(f"cutoff checked: {now_cet.hour >= cutoff_hour}")
    if not is_weekday:
        print(f"\u23f8 Weekend ({now_cet.strftime('%A')}) — not triggering next run.")
        dbutils.notebook.exit("DONE_WEEKEND")
    elif now_cet.hour >= cutoff_hour or now_cet.hour < start_hour :
        print(f"\u23f8 Not in execution window {start_hour}:00 to {cutoff_hour}:00 CET ({now_cet.strftime('%H:%M')}) — not triggering next run.")
        dbutils.notebook.exit("DONE_END_OF_DAY")
    else:
        print(f"\u23f8 Inside execution window {start_hour}:00 to  {cutoff_hour}:00 CET ({now_cet.strftime('%H:%M')}) — continue with next run.")
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        print(f"Request Token: {token}")
        response = req.post(
            f"https://{workspace_url}/api/2.1/jobs/run-now",
            headers={"Authorization": f"Bearer {token}"},
            json={"job_id": int(job_id)}
        )

        if response.status_code == 200:
            next_run_id = response.json().get("run_id")
            print(f"\u2713 Self-triggered next run: run_id={next_run_id} (time: {now_cet.strftime('%H:%M')} CET)")
        else:
            print(f"\u26a0 Failed to trigger next run: {response.status_code} - {response.text}")
else:
    print("Not running as a job — skipping self-trigger.")